In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer # Text -> Numeric
from sklearn.cluster import KMeans
from umap import UMAP # Dimensionality Reduction for Visualization
import plotly.express as px  # Interactive Visualization

/Users/jadentyh/Desktop/IS455/Project/OPE/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Gather review/text data
kol_posts = pd.read_csv("../data/kol/KOL_Posts.csv")
rest = pd.read_parquet("../data/mv_dataset_parquet/restaurants.parquet")
placesapinew = pd.read_csv("../_1_eda/places_api_new_results.csv")
rest_agg = pd.read_parquet("../_1_eda/restaurant_agg.parquet")
eda_valid_bookings = pd.read_parquet("../_1_eda/valid_bookings.parquet")

latest = pd.read_parquet("../_2_feature_engineering+momentum/start/latest.parquet")
priority = pd.read_parquet("../_2_feature_engineering+momentum/start/priority_latest_momentum_labels.parquet")
val_bookings = pd.read_parquet("../_2_feature_engineering+momentum/start/valid_bookings_with_currency_and_google_restaurants_without_duplicates.parquet")
valid_bookings = pd.read_parquet('../_2_feature_engineering+momentum/start/valid_bookings.parquet')

rest_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews.parquet")
rest_best_reviews = pd.read_parquet("../_3_marketing/restaurant_reviews_all_best.parquet")

eda_valid_bookings.columns

Index(['has_promptpay', 'has_cc', 'has_shopee_pay', 'has_special_request',
       'medium', 'created_at', 'prepared', 'refund', 'adjusted', 'ack', 'id',
       'restaurant_id', 'is_temporary', 'for_locking_system', 'no_show',
       'arrived', 'channel', 'user_id_masked', 'kids', 'adult', 'end_time',
       'start_time', 'active', 'revenue_dollars', 'booking_date',
       'day_of_week_index', 'day_of_week', 'is_outlier', 'outlier_reason'],
      dtype='str')

In [3]:
rest.shape #2474
rest_agg.shape # 3656
valid_bookings.shape # 535 899
priority.shape # 1693
rest_reviews.shape # 9674, 15
rest_best_reviews.shape # 76690, 12
valid_bookings.head()
rest_reviews['input_restaurant_name'].nunique() #1947
rest_best_reviews['input_restaurant_name'].nunique() # 1892
eda_valid_bookings.shape # 535 899

kol_posts.shape # 923 35
placesapinew.shape # 2392 12
kol_posts['Restaurant Code'].nunique() # 236
kol_posts['Restaurant Name'].nunique() # 301
placesapinew['input_string'].nunique() # 2392
eda_valid_bookings['restaurant_id'].nunique() # 3210
val_bookings.shape # 241 311
val_bookings['restaurant_id'].nunique() # 1737
val_bookings.columns

Index(['has_promptpay', 'has_cc', 'has_shopee_pay', 'has_special_request',
       'medium', 'created_at', 'prepared', 'refund', 'adjusted', 'ack', 'id',
       'restaurant_id', 'is_temporary', 'for_locking_system', 'no_show',
       'arrived', 'channel', 'user_id_masked', 'kids', 'adult', 'end_time',
       'start_time', 'active', 'revenue_dollars', 'booking_date',
       'day_of_week_index', 'day_of_week', 'is_outlier', 'outlier_reason',
       'name', 'days_in_advance', 'input_string', 'found', 'official_name',
       'city', 'country', 'formatted_address', 'rating', 'website', 'error',
       'Cuisine', 'Cuisine_confidence', 'raw_types', 'revenue_thb',
       'total_guests'],
      dtype='str')

In [4]:
# Aggregate KOL Posts per restaurant
kol_aggregated = (
    kol_posts.groupby('Restaurant Name')['Content']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index()
)
kol_aggregated.columns = ['Restaurant Name', 'kol_text']

print('kol_aggregated shape:', kol_aggregated.shape) # Shape = (301, 2)

placesapinew['google_text'] = ( # Combine relevant fields from placesapinew into one text string for each restaurant
        placesapinew['input_string'].fillna('') + ' ' +
        placesapinew['raw_types'].fillna('').str.replace(',', ' ') + ' ' +
        placesapinew['Cuisine'].fillna('')
)

# Merge KOL aggregated text with placesapinew to get a combined dataset for clustering
kol_restaurant_mapping = kol_posts[['Restaurant Name', 'Restaurant Code']].drop_duplicates()
places_merged = placesapinew.merge(
    kol_restaurant_mapping,
    left_on='input_string', # Assuming 'input_string' in placesapinew corresponds to 'Restaurant Name' in kol_posts
    right_on='Restaurant Name', # Merge on restaurant name
    how='left'
)

# Merge dataset with both the google text and the KOL text for each restaurant
places_aggregated = places_merged[['Restaurant Name', 'google_text']].rename(
    columns={'Restaurant Name': 'restaurant name'}
) # Shape = 2395

# Aggregate rest reviews per restaurant
rest_reviews_agg = (
    rest_reviews.groupby('input_restaurant_name')['review_text']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index()
)
rest_reviews_agg.columns = ['input_restaurant_name', 'rest_reviews']
print('rest_reviews_agg shape:', rest_reviews_agg.shape) # Shape = (1947, 2)

rest_places_merged = rest_reviews_agg.merge( # Merge restaurant reviews with the aggregated places data to get a combined dataset for clustering
    places_aggregated,
    left_on='input_restaurant_name', # Assuming 'input_restaurant_name' in rest_reviews corresponds to 'restaurant name' in places_aggregated
    right_on='restaurant name', # Merge on restaurant name
    how='left'
)
print('rest_places_merged shape:', rest_places_merged.shape) # Shape = (1950, 3)

merge_id = rest_places_merged = rest_places_merged.merge( # Merge restaurant reviews with the original restaurant data to get restaurant_id for merging with bookings
    rest,
    left_on='input_restaurant_name', # Assuming 'input_restaurant_name' in rest_reviews corresponds to 'name' in rest
    right_on='name', # Merge on restaurant name
    how='left'
)
print('merge id shape:', merge_id.shape) # Shape = (1950, 7)

# Aggregate valid bookings per restaurant
val_bookings_agg = val_bookings.groupby('restaurant_id').count()
val_bookings_agg.reset_index(inplace=True)
print('val_bookings_agg shape:', val_bookings_agg.shape) # Shape = (1737, 7)

id_bookings = merge_id.merge( # Merge the combined restaurant reviews and places data with the valid bookings data to get the final dataset for clustering
    val_bookings_agg,
    left_on='restaurant_id', # Assuming 'restaurant_id' in rest corresponds to 'restaurant_id' in eda_valid_bookings
    right_on='restaurant_id', # Merge on restaurant_id
    how='left'
)
print('id_bookings shape:', id_bookings.shape) # Shape = (1950, 7)
print('Unique restaurants in id_bookings:', id_bookings['input_restaurant_name'].nunique()) # 1947
id_bookings

kol_aggregated shape: (301, 2)
rest_reviews_agg shape: (1947, 2)
rest_places_merged shape: (1950, 4)
merge id shape: (1950, 7)
val_bookings_agg shape: (1737, 45)
id_bookings shape: (1950, 51)
Unique restaurants in id_bookings: 1947


,input_restaurant_name,rest_reviews,restaurant name,google_text,restaurant_id,name_x,days_in_advance_x,has_promptpay,has_cc,has_shopee_pay,...,country,formatted_address,rating,website,error,Cuisine,Cuisine_confidence,raw_types,revenue_thb,total_guests
0,&T modern Thai grill,-The restaurant is beautifully decorated with ...,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,"The drinks and snacks are delicious, good tast...",NaN,NaN,3379.0,'@RICE Restaurant by At Rice Resort (Nakhon Na...,90.0,3.0,5.0,14.0,...,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0
2,100 Degrees Hotpot Central Chaengwattana,"Delicious, good price, great value. This was m...",NaN,NaN,6420.0,100 Degrees Hotpot Central Chaengwattana,90.0,0.0,5.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0
3,100 Degrees Hotpot Cosmo Bazaar,Kimchi soup is the best! So delicious. I've ea...,NaN,NaN,6421.0,100 Degrees Hotpot Cosmo Bazaar,90.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,123 ICHI NI SAN Sathorn Soi 1,"The food here is delicious, and they have gre...",NaN,NaN,2724.0,123 ICHI NI SAN Sathorn Soi 1,90.0,1.0,1.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,sala ayutthaya staycation (Ayutthaya),A gem!\n\n2 nights / 2 breakfasts / 2 dinners...,NaN,NaN,5916.0,sala ayutthaya staycation (Ayutthaya),90.0,2.0,1.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
1946,sala bang pa-in Staycation (Ayutthaya),"This stunning resort hotel, like a paradise on...",NaN,NaN,5917.0,sala bang pa-in Staycation (Ayutthaya),90.0,2.0,1.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
1947,อร่อย Together,Went to sing karaoke and was pleasantly surpri...,NaN,NaN,6635.0,อร่อย Together,30.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1948,中国好味道 China Taste,Food is relatively less salty and more home co...,中国好味道 China Taste,中国好味道 China Taste restaurant asian_restaurant ...,5899.0,中国好味道 China Taste,90.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# 2. Prepare text data

# Combine both rest reviews and google text columns into one
id_bookings['text'] = (
    id_bookings['rest_reviews'].fillna('') + ' ' +
    id_bookings['google_text'].fillna('')
)
reviews = id_bookings.drop(columns=['rest_reviews', 'google_text', 'restaurant name', 'name_x']) # Keep only the combined text and relevant identifiers for clustering
reviews

,input_restaurant_name,restaurant_id,days_in_advance_x,has_promptpay,has_cc,has_shopee_pay,has_special_request,medium,created_at,prepared,...,formatted_address,rating,website,error,Cuisine,Cuisine_confidence,raw_types,revenue_thb,total_guests,text
0,&T modern Thai grill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-The restaurant is beautifully decorated with ...
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,3379.0,90.0,3.0,5.0,14.0,14.0,14.0,14.0,14.0,...,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,"The drinks and snacks are delicious, good tast..."
2,100 Degrees Hotpot Central Chaengwattana,6420.0,90.0,0.0,5.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,"Delicious, good price, great value. This was m..."
3,100 Degrees Hotpot Cosmo Bazaar,6421.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kimchi soup is the best! So delicious. I've ea...
4,123 ICHI NI SAN Sathorn Soi 1,2724.0,90.0,1.0,1.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,"The food here is delicious, and they have gre..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,sala ayutthaya staycation (Ayutthaya),5916.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,A gem!\n\n2 nights / 2 breakfasts / 2 dinners...
1946,sala bang pa-in Staycation (Ayutthaya),5917.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,"This stunning resort hotel, like a paradise on..."
1947,อร่อย Together,6635.0,30.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,Went to sing karaoke and was pleasantly surpri...
1948,中国好味道 China Taste,5899.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Food is relatively less salty and more home co...


In [6]:
print("Unique KOL restaurants:", kol_posts['Restaurant Name'].nunique())
print("After merge:", places_merged['Restaurant Name'].nunique())

print("Unique review restaurants:", rest_reviews['input_restaurant_name'].nunique())
print("Final merged:", reviews['input_restaurant_name'].nunique())

#rest_places_merged[rest_places_merged['input_restaurant_name'] == 'JW Cafe at JW Marriott Hotel Bangkok']
#rest_places_merged[rest_places_merged['input_restaurant_name'] == 'Melody Bangkok']
#rest[rest['name'] == 'JW Cafe at JW Marriott Hotel Bangkok']

Unique KOL restaurants: 301
After merge: 197
Unique review restaurants: 1947
Final merged: 1947


In [7]:
# These are Google Places API tags that leaked into your text column.
# They are NOT customer perception language.
# Remove them before TF-IDF.

redundant_words = [
    'point_of_interest', 'establishment', 'general',
    'food establishment', 'restaurant point_of_interest',
    'food point_of_interest', 'point_of_interest establishment',
    'establishment general', 'restaurant food',
    # Also remove raw Google types that commonly appear:
    'meal_delivery', 'meal_takeaway', 'lodging',
    'tourist_attraction', 'night_club', 'shopping_mall',
    'Bangkok', 'sukhumvit', 'singapore', 'bangkok', 'sukhumvit', 'Singapore', 'thai'
    'baht', 'order', 'ordered', 'dishes', 'fried', 'sauce', 'spicy', 
    'time', 'didn', 'eat', 'meat', 'restaurant', 'soup', 'super', 'really',
    'terrible', 'wasn', 'gr', 'floor',
    'rice', 'pork', 'beef', 'chicken', 'crab'
]

import re

def remove_redundant_words(text):
    """Remove redundant words from review text."""
    cleaned = text
    # Remove multi-word junk phrases first (longer matches first)
    for phrase in sorted(redundant_words, key=len, reverse=True):
        cleaned = cleaned.replace(phrase, ' ')
    # Collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

reviews['text_clean'] = reviews['text'].apply(remove_redundant_words)

In [8]:
# 3. Text vectorization

vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2
)
text_vectors = vectorizer.fit_transform(reviews['text_clean'])

# Verify the junk is gone:
feature_names = vectorizer.get_feature_names_out()
junk_remaining = [f for f in feature_names if 'establishment' in f or 'point_of_interest' in f]
print(f"Junk terms remaining: {junk_remaining}")  # Should be empty

Junk terms remaining: []


In [9]:
"""
Pick K by checking: are all K clusters actually DIFFERENT from each other?
If two clusters have the same distinctive terms, K is too high.
"""
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

feature_names = vectorizer.get_feature_names_out()

for k in [5, 6, 7, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(text_vectors)
    
    print(f"\n{'='*70}")
    print(f"K = {k}")
    print(f"{'='*70}")
    
    cluster_top_terms = {}
    for c in range(k):
        centroid = km.cluster_centers_[c]
        global_avg = km.cluster_centers_.mean(axis=0)
        diff = centroid - global_avg
        top_indices = diff.argsort()[-8:][::-1]
        terms = set(feature_names[i] for i in top_indices)
        cluster_top_terms[c] = terms
        
        mask = labels == c
        print(f"  Cluster {c} ({mask.sum():>4} posts): {', '.join(sorted(terms))}")
    
    # Check overlap: how many clusters share >50% of their top terms?
    duplicates = 0
    for i in range(k):
        for j in range(i+1, k):
            overlap = len(cluster_top_terms[i] & cluster_top_terms[j])
            total = len(cluster_top_terms[i] | cluster_top_terms[j])
            if overlap / total > 0.4:
                duplicates += 1
    
    print(f"\n  → Redundant cluster pairs (>40% term overlap): {duplicates}")
    print(f"  → {'✅ Good — clusters are distinct' if duplicates == 0 else '⚠️ Some clusters are too similar, K may be too high'}")


K = 5
  Cluster 0 ( 589 posts): amazing, bar, dinner, drinks, experience, friendly, nice, place
  Cluster 1 ( 337 posts): atmosphere, delicious, delicious food, food, food delicious, good, parking, reasonable
  Cluster 2 ( 658 posts): baht, buffet, customers, don, just, like, quality, table
  Cluster 3 ( 251 posts): excellent, excellent service, good service, provided, provided excellent, service, service excellent, staff
  Cluster 4 ( 115 posts): breakfast, buffet, coffee, day, hotel, location, room, view

  → Redundant cluster pairs (>40% term overlap): 0
  → ✅ Good — clusters are distinct

K = 6
  Cluster 0 ( 347 posts): authentic, chef, curry, dish, experience, highly, meal, thai
  Cluster 1 ( 389 posts): bar, cafe, coffee, drinks, good, music, nice, place
  Cluster 2 ( 114 posts): breakfast, buffet, day, hotel, located, location, room, view
  Cluster 3 ( 311 posts): attentive, excellent, excellent service, experience, provided, provided excellent, service, staff
  Cluster 4 ( 478

In [10]:
# Cluster Posts (Initial Clustering at Post Level)
# We still decided to stick to 5 clusters
c = 5
print("\n" + "="*70)
print("STEP 3: Clustering Posts (K-Means with 5 Clusters)")
print("="*70)

kmeans = KMeans(n_clusters=c, random_state=42, n_init=10)
clusters = kmeans.fit_predict(text_vectors)
reviews['cluster'] = clusters

print(f"Assigned {len(reviews)} posts to {c} clusters")
print("\nPost distribution across clusters:")
print(reviews['cluster'].value_counts().sort_index())


STEP 3: Clustering Posts (K-Means with 5 Clusters)
Assigned 1950 posts to 5 clusters

Post distribution across clusters:
cluster
0    589
1    337
2    658
3    251
4    115
Name: count, dtype: int64


In [11]:
# Define Predefined Themes according to results above
cluster_themes = {
    0: "Bar & Alcoholic Drinks",
    1: "Good food and atmosphere",
    2: "Buffet & premium dining",
    3: "Excellent service",
    4: "Cafes, Coffee, Breakfast"
}

reviews['theme'] = reviews['cluster'].map(cluster_themes)

In [12]:
reviews.to_csv("reviews.csv")

In [13]:
# Validate Cluster Themes (Check Keywords)
print("\n" + "="*70)
print("STEP 4: Validating Cluster Themes (Auto vs Manual)")
print("="*70)

for cluster_id in range(5):
    cluster_texts = reviews[reviews['cluster'] == cluster_id]['text']
    
    if len(cluster_texts) > 0:
        cluster_vectorizer = TfidfVectorizer(max_features=10, stop_words='english')
        cluster_vectorizer.fit(cluster_texts)
        top_terms = cluster_vectorizer.get_feature_names_out()
        
        print(f"\n Cluster {cluster_id} → {cluster_themes[cluster_id]}")
        print(f"    Posts: {len(cluster_texts)} ({len(cluster_texts)/len(reviews)*100:.1f}%)")
        print(f"    Auto-detected keywords: {', '.join(top_terms)}")
        print(f"    Check if keywords match your theme name!")

reviews

# # MULTI-THEME RESTAURANT ASSIGNMENT
# print("\n" + "="*70)
# print("STEP 5: Aggregating to Restaurant Level (Multi-Theme Method)")
# print("="*70)

# # Count posts per restaurant per cluster
# restaurant_theme_counts = (
#     reviews.groupby(['restaurant name', 'cluster', 'theme'])
#     .size()
#     .reset_index(name='post_count')
# )

# # Get total posts per restaurant
# restaurant_post_totals = reviews.groupby('restaurant name').size().reset_index(name='total_posts')

# # Merge to calculate percentages
# restaurant_theme_counts = restaurant_theme_counts.merge(
#     restaurant_post_totals, 
#     on='restaurant name'
# )

# restaurant_theme_counts['percentage'] = ( # Calculate percentage of posts for each theme per restaurant
#     restaurant_theme_counts['post_count'] / restaurant_theme_counts['total_posts'] * 100
# )

# print(f"\n Analyzing {len(restaurant_post_totals)} unique restaurants...")


STEP 4: Validating Cluster Themes (Auto vs Manual)

 Cluster 0 → Bar & Alcoholic Drinks
    Posts: 589 (30.2%)
    Auto-detected keywords: atmosphere, delicious, food, good, great, nice, place, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 1 → Good food and atmosphere
    Posts: 337 (17.3%)
    Auto-detected keywords: atmosphere, delicious, food, good, great, nice, price, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 2 → Buffet & premium dining
    Posts: 658 (33.7%)
    Auto-detected keywords: delicious, dishes, food, good, great, like, price, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 3 → Excellent service
    Posts: 251 (12.9%)
    Auto-detected keywords: atmosphere, delicious, excellent, food, good, great, provided, restaurant, service, staff
    Check if keywords match your theme name!

 Cluster 4 → Cafes, Coffee, Breakfast
    Posts: 115 (5.9%)
    Auto-detected keyword

,input_restaurant_name,restaurant_id,days_in_advance_x,has_promptpay,has_cc,has_shopee_pay,has_special_request,medium,created_at,prepared,...,error,Cuisine,Cuisine_confidence,raw_types,revenue_thb,total_guests,text,text_clean,cluster,theme
0,&T modern Thai grill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-The restaurant is beautifully decorated with ...,-The is beautifully decorated with a nice atmo...,2,Buffet & premium dining
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,3379.0,90.0,3.0,5.0,14.0,14.0,14.0,14.0,14.0,...,14.0,14.0,14.0,14.0,14.0,14.0,"The drinks and snacks are delicious, good tast...","The drinks and snacks are delicious, good tast...",1,Good food and atmosphere
2,100 Degrees Hotpot Central Chaengwattana,6420.0,90.0,0.0,5.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,"Delicious, good price, great value. This was m...","Delicious, good p , value. This was my first i...",1,Good food and atmosphere
3,100 Degrees Hotpot Cosmo Bazaar,6421.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Kimchi soup is the best! So delicious. I've ea...,Kimchi is the best! So delicious. I've en it m...,1,Good food and atmosphere
4,123 ICHI NI SAN Sathorn Soi 1,2724.0,90.0,1.0,1.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,"The food here is delicious, and they have gre...","The food here is delicious, and they have deal...",3,Excellent service
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,sala ayutthaya staycation (Ayutthaya),5916.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,A gem!\n\n2 nights / 2 breakfasts / 2 dinners...,A gem! 2 nights / 2 breakfasts / 2 dinners Pro...,4,"Cafes, Coffee, Breakfast"
1946,sala bang pa-in Staycation (Ayutthaya),5917.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,"This stunning resort hotel, like a paradise on...","This stunning resort hotel, like a paradise on...",4,"Cafes, Coffee, Breakfast"
1947,อร่อย Together,6635.0,30.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,Went to sing karaoke and was pleasantly surpri...,Went to sing karaoke and was pleasantly surpri...,1,Good food and atmosphere
1948,中国好味道 China Taste,5899.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Food is relatively less salty and more home co...,Food is relatively less salty and more home co...,0,Bar & Alcoholic Drinks


In [14]:
# # Keep themes with at least 20% of restaurant's posts
# THRESHOLD = 0  # Adjust this threshold (e.g., 15, 20, 25, 30)

# significant_themes = restaurant_theme_counts[
#     restaurant_theme_counts['percentage'] >= THRESHOLD
# ].copy()

# print(f"\n Keeping themes that represent ≥{THRESHOLD}% of a restaurant's posts")
# print(f"   Found {len(significant_themes)} restaurant-theme combinations")

# # Create multi-theme labels
# restaurant_multi_themes = (
#     significant_themes.groupby('restaurant name')
#     .agg({
#         'theme': lambda x: ' + '.join(sorted(x)),  # Combine themes
#         'cluster': lambda x: list(x),  # Keep cluster IDs
#         'percentage': lambda x: list(x)  # Keep percentages
#     })
#     .reset_index()
# )

# restaurant_multi_themes.columns = ['restaurant name', 'combined_themes', 'clusters', 'percentages']

# print("\n" + "="*70)
# print("RESTAURANT THEME DISTRIBUTION (Multi-Theme Assignment)")
# print("="*70)
# print(restaurant_multi_themes['combined_themes'].value_counts().head(15))

# # Also create primary theme (for visualization)
# # Assign each restaurant to its DOMINANT theme (highest percentage)
# restaurant_primary_theme = (
#     restaurant_theme_counts
#     .sort_values('percentage', ascending=False)
#     .groupby('restaurant name')
#     .first()
#     .reset_index()
# )

# restaurant_primary_theme = restaurant_primary_theme[[
#     'restaurant name', 'cluster', 'theme', 'percentage', 'post_count', 'total_posts'
# ]]

# print("\n" + "="*70)
# print("PRIMARY THEME DISTRIBUTION (Dominant Theme per Restaurant)")
# print("="*70)
# print(restaurant_primary_theme['theme'].value_counts())

In [15]:
# # PRIMARY THEME ONLY (no multi-theme)

# print("\n" + "="*70)
# print("PRIMARY THEME ASSIGNMENT (Dominant Theme per Restaurant)")
# print("="*70)

# # For each restaurant, pick thes theme with highest percentage
# # If ties happen, this keeps the first after sorting (stable + deterministic if extra sort columns added)
# restaurant_primary_theme = (
#     restaurant_theme_counts
#     .sort_values(
#         ['restaurant name', 'percentage', 'post_count'],
#         ascending=[True, False, False]
#     )
#     .groupby('restaurant name', as_index=False)
#     .first()
# )

# # # Keep relevant columns
# # restaurant_primary_theme = restaurant_primary_theme[
# #     ['restaurant name', 'cluster', 'theme', 'percentage', 'post_count', 'total_posts']
# # ]

# print(f"Total restaurants assigned a primary theme: {restaurant_primary_theme['restaurant name'].nunique()}")
# print("\nPrimary theme distribution:")
# print(restaurant_primary_theme['theme'].value_counts())

In [16]:
# STEP 7: Create Review-Level Embeddings for Visualization
print("\n" + "="*70)
print("STEP 7: Creating Review-Level Embeddings")
print("="*70)

# Reset index for clean alignment
reviews_reset = reviews.reset_index(drop=True)

# Use TF-IDF vectors directly at review level
review_embeddings = np.array(text_vectors.toarray())  # if sparse matrix
review_labels = reviews_reset['input_restaurant_name'].tolist()  # optional metadata

print(f"Created embeddings for {len(review_embeddings)} reviews")
print(f"Embedding shape: {review_embeddings.shape}")


STEP 7: Creating Review-Level Embeddings
Created embeddings for 1950 reviews
Embedding shape: (1950, 200)


In [17]:
# STEP 8: Dimensionality Reduction (200D → 2D using UMAP)
print("\n" + "="*70)
print("STEP 7: Reducing Dimensions (200D → 2D for Visualization)")
print("="*70)

# Can experiment with UMAP parameters like n_neighbors and min_dist to see how it affects the clustering in 2D space
reducer = UMAP(
    n_components=2,
    n_neighbors=30,    # ← Increase from 15 (preserves more global structure)
    min_dist=1,      # ← Increase from 0.1 (allows more spread)
    metric='cosine',   # ← Use cosine similarity (better for text)
    random_state=42
)
embeddings_2d = reducer.fit_transform(review_embeddings) # restaurant_embeddings -> review_embeddings
print(f" Reduced to 2D coordinates")


STEP 7: Reducing Dimensions (200D → 2D for Visualization)


/Users/jadentyh/Desktop/IS455/Project/OPE/venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


 Reduced to 2D coordinates


In [18]:
viz_df = reviews.copy()

viz_df['UMAP Component 1 (Semantic Similarity Axis)'] = embeddings_2d[:, 0]
viz_df['UMAP Component 2 (Theme Variation Axis)'] = embeddings_2d[:, 1]

print(f"\n Prepared {len(viz_df)} restaurants for visualization")


 Prepared 1950 restaurants for visualization


In [19]:
# # Create Visualization DataFrame
# viz_df = pd.DataFrame({
#     'restaurant name': restaurant_names_list, # Align restaurant names with the order of embeddings
#     'UMAP Component 1 (Semantic Similarity Axis)': embeddings_2d[:, 0], # UMAP component 1 captures the main semantic differences between restaurants based on their reviews and google text
#     'UMAP Component 2 (Theme Variation Axis)': embeddings_2d[:, 1], # UMAP component 2 captures the variation in themes among restaurants
# })

# # Merge with primary theme (for color)
# viz_df = viz_df.merge( # Merge primary theme info for coloring the points in the visualization
#     restaurant_primary_theme[['restaurant name', 'cluster', 'theme', 'total_posts', 'percentage']], 
#     on='restaurant name'
# )

# # Merge with multi-themes (for hover info)
# viz_df = viz_df.merge(
#     restaurant_multi_themes[['restaurant name', 'combined_themes']], 
#     on='restaurant name',
#     how='left'
# )

# # Fill single-theme restaurants
# viz_df['combined_themes'] = viz_df['combined_themes'].fillna(viz_df['theme'])

# # Rename for clarity
# viz_df = viz_df.rename(columns={
#     'theme': 'Primary Theme',
#     'total_posts': 'Number of Posts',
#     'percentage': 'Primary Theme %',
#     'combined_themes': 'All Themes'
# })

# print(f"\n Prepared {len(viz_df)} restaurants for visualization")

In [20]:
# Create Interactive Plotly Visualization
print("\n" + "="*70)
print("STEP 8: Creating Interactive Visualization")
print("="*70)

fig = px.scatter( # Create an interactive scatter plot with Plotly Express
    viz_df,
    x='UMAP Component 1 (Semantic Similarity Axis)',
    y='UMAP Component 2 (Theme Variation Axis)',
    color='theme',  # Color points by primary theme
    #size='Number of Posts',  # Bigger dots = more posts about this restaurant # Comment out if you want uniform dot sizes
    hover_data={ # Show these details when hovering over a restaurant
        'input_restaurant_name': True,
        'theme': True,
        'UMAP Component 1 (Semantic Similarity Axis)': ':.2f',
        'UMAP Component 2 (Theme Variation Axis)': ':.2f'
    },
    title='<b>Restaurant Segmentation by Customer Perception Themes</b><br>' + 
          '<sub>(1 Dot = 1 Restaurant | Color = Primary Theme)</sub>', # | Size = Number of Posts (Commented out for uniform sizes)
    color_discrete_sequence=px.colors.qualitative.Set3,
)

fig.update_layout( # Customize layout for better readability and aesthetics
    font=dict(size=12),
    title_font=dict(size=16),
    width=1400,
    height=800,
    legend=dict(
        title="Primary Customer Perception Theme",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    hovermode='closest'
)

# # Update marker size range for better visibility
# fig.update_traces(
#     marker=dict(
#         sizemode='diameter',
#         sizemin=5,
#         sizeref=2.*max(viz_df['Number of Posts'])/(40.**2),
#         line=dict(width=1, color='DarkSlateGrey')
#     )
# )

# Add cluster centroids with labels
centroids = viz_df.groupby("theme")[
    ['UMAP Component 1 (Semantic Similarity Axis)', 
     'UMAP Component 2 (Theme Variation Axis)']
].mean().reset_index()

for _, row in centroids.iterrows():
    fig.add_annotation(
        x=row['UMAP Component 1 (Semantic Similarity Axis)'],
        y=row['UMAP Component 2 (Theme Variation Axis)'],
        text=f"<b>{row['theme']}</b>",
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="black",
        borderwidth=1.5,
        borderpad=5
    )

fig.write_html("restaurant_clusters.html")
print("\n Visualization saved to 'restaurant_clusters.html'")


STEP 8: Creating Interactive Visualization

 Visualization saved to 'restaurant_clusters.html'


In [21]:
# Save Results to CSV
print("\n" + "="*70)
print("STEP 9: Saving Results")
print("="*70)

viz_df.to_csv('clustering_results.csv', index=False)
print(" Saved 'clustering_results.csv'")
viz_df


STEP 9: Saving Results
 Saved 'clustering_results.csv'


,input_restaurant_name,restaurant_id,days_in_advance_x,has_promptpay,has_cc,has_shopee_pay,has_special_request,medium,created_at,prepared,...,Cuisine_confidence,raw_types,revenue_thb,total_guests,text,text_clean,cluster,theme,UMAP Component 1 (Semantic Similarity Axis),UMAP Component 2 (Theme Variation Axis)
0,&T modern Thai grill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-The restaurant is beautifully decorated with ...,-The is beautifully decorated with a nice atmo...,2,Buffet & premium dining,-16.651752,-11.320339
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,3379.0,90.0,3.0,5.0,14.0,14.0,14.0,14.0,14.0,...,14.0,14.0,14.0,14.0,"The drinks and snacks are delicious, good tast...","The drinks and snacks are delicious, good tast...",1,Good food and atmosphere,-16.129368,-11.529309
2,100 Degrees Hotpot Central Chaengwattana,6420.0,90.0,0.0,5.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,"Delicious, good price, great value. This was m...","Delicious, good p , value. This was my first i...",1,Good food and atmosphere,-17.105635,-11.440622
3,100 Degrees Hotpot Cosmo Bazaar,6421.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Kimchi soup is the best! So delicious. I've ea...,Kimchi is the best! So delicious. I've en it m...,1,Good food and atmosphere,-16.645699,-11.471118
4,123 ICHI NI SAN Sathorn Soi 1,2724.0,90.0,1.0,1.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,"The food here is delicious, and they have gre...","The food here is delicious, and they have deal...",3,Excellent service,-16.838720,-11.000373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,sala ayutthaya staycation (Ayutthaya),5916.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,A gem!\n\n2 nights / 2 breakfasts / 2 dinners...,A gem! 2 nights / 2 breakfasts / 2 dinners Pro...,4,"Cafes, Coffee, Breakfast",-9.658490,-11.401080
1946,sala bang pa-in Staycation (Ayutthaya),5917.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,"This stunning resort hotel, like a paradise on...","This stunning resort hotel, like a paradise on...",4,"Cafes, Coffee, Breakfast",-10.103704,-10.732664
1947,อร่อย Together,6635.0,30.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,Went to sing karaoke and was pleasantly surpri...,Went to sing karaoke and was pleasantly surpri...,1,Good food and atmosphere,-15.752746,-6.670656
1948,中国好味道 China Taste,5899.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Food is relatively less salty and more home co...,Food is relatively less salty and more home co...,0,Bar & Alcoholic Drinks,-8.708611,-4.885527


In [22]:
# Summary Statistics
print("\n" + "="*70)
print(" SUMMARY STATISTICS")
print("="*70)

print(f"\n Total Restaurants: {len(viz_df)}")
# print(f" Total Posts: {viz_df['Number of Posts'].sum()}")
# print(f" Avg Posts per Restaurant: {viz_df['Number of Posts'].mean():.1f}")

# print("\n Single-Theme vs Multi-Theme Restaurants:")
# single_theme = viz_df[viz_df['All Themes'] == viz_df['Primary Theme']]
# multi_theme = viz_df[viz_df['All Themes'] != viz_df['Primary Theme']]

# print(f"   Single-theme restaurants: {len(single_theme)} ({len(single_theme)/len(viz_df)*100:.1f}%)")
# print(f"   Multi-theme restaurants: {len(multi_theme)} ({len(multi_theme)/len(viz_df)*100:.1f}%)")

# print("\n Most Common Multi-Theme Combinations:")
# if len(multi_theme) > 0:
#     print(multi_theme['All Themes'].value_counts().head(10))

print("\n CLUSTERING COMPLETE with new data!")
print("="*70)


 SUMMARY STATISTICS

 Total Restaurants: 1950

 CLUSTERING COMPLETE with new data!


In [23]:
# # Merge with restaurant to get restaurant id
# out_merge_id = pd.merge(
#     output_df,
#     rest_agg,
#     left_on="restaurant name",
#     right_on="name",
#     how="left"
# )

# viz_merge_id = pd.merge(
#     viz_df,
#     rest_agg,
#     left_on="restaurant name",
#     right_on="name",
#     how="left"
# )

# clustering_results.to_csv('clustering_results.csv', index=False)

In [24]:
viz_df['cluster'].isnull().sum()
viz_df[viz_df['input_restaurant_name']=='Myste']
viz_df['input_restaurant_name'].nunique()
viz_df

,input_restaurant_name,restaurant_id,days_in_advance_x,has_promptpay,has_cc,has_shopee_pay,has_special_request,medium,created_at,prepared,...,Cuisine_confidence,raw_types,revenue_thb,total_guests,text,text_clean,cluster,theme,UMAP Component 1 (Semantic Similarity Axis),UMAP Component 2 (Theme Variation Axis)
0,&T modern Thai grill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-The restaurant is beautifully decorated with ...,-The is beautifully decorated with a nice atmo...,2,Buffet & premium dining,-16.651752,-11.320339
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,3379.0,90.0,3.0,5.0,14.0,14.0,14.0,14.0,14.0,...,14.0,14.0,14.0,14.0,"The drinks and snacks are delicious, good tast...","The drinks and snacks are delicious, good tast...",1,Good food and atmosphere,-16.129368,-11.529309
2,100 Degrees Hotpot Central Chaengwattana,6420.0,90.0,0.0,5.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,"Delicious, good price, great value. This was m...","Delicious, good p , value. This was my first i...",1,Good food and atmosphere,-17.105635,-11.440622
3,100 Degrees Hotpot Cosmo Bazaar,6421.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Kimchi soup is the best! So delicious. I've ea...,Kimchi is the best! So delicious. I've en it m...,1,Good food and atmosphere,-16.645699,-11.471118
4,123 ICHI NI SAN Sathorn Soi 1,2724.0,90.0,1.0,1.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,"The food here is delicious, and they have gre...","The food here is delicious, and they have deal...",3,Excellent service,-16.838720,-11.000373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,sala ayutthaya staycation (Ayutthaya),5916.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,A gem!\n\n2 nights / 2 breakfasts / 2 dinners...,A gem! 2 nights / 2 breakfasts / 2 dinners Pro...,4,"Cafes, Coffee, Breakfast",-9.658490,-11.401080
1946,sala bang pa-in Staycation (Ayutthaya),5917.0,90.0,2.0,1.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,"This stunning resort hotel, like a paradise on...","This stunning resort hotel, like a paradise on...",4,"Cafes, Coffee, Breakfast",-10.103704,-10.732664
1947,อร่อย Together,6635.0,30.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,Went to sing karaoke and was pleasantly surpri...,Went to sing karaoke and was pleasantly surpri...,1,Good food and atmosphere,-15.752746,-6.670656
1948,中国好味道 China Taste,5899.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Food is relatively less salty and more home co...,Food is relatively less salty and more home co...,0,Bar & Alcoholic Drinks,-8.708611,-4.885527


In [25]:
# STEP 10: Export dashboard artifacts for Streamlit (UPDATED)
# - Ensures a cluster map is available on the frontend by exporting x/y coordinates.
# - Fixes column naming so loader.py can always find: restaurant_id, name, cluster_id, cluster_label, x, y
# - Uses viz_df (UMAP restaurant-level map) as the primary source for x/y.
# - Falls back gracefully if some fields are missing.

from pathlib import Path
import numpy as np
import pandas as pd

print("\n" + "="*70)
print("STEP 10: Export dashboard artifacts for Streamlit (UPDATED)")
print("="*70)

def _find_project_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "frontend_dashboard").exists():
            return path
    return Path.cwd()

project_root = _find_project_root()
export_dir = project_root / "frontend_dashboard" / "data" / "clustering"
export_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) restaurant_cluster_assignments.parquet
#    (THIS is what powers the cluster map)
# -----------------------------
assignments = pd.DataFrame()

# Prefer viz_df because it contains UMAP x/y
if "viz_df" in globals() and isinstance(viz_df, pd.DataFrame) and len(viz_df):
    #keep_base = ["input_restaurant_name", "cluster", "Primary Theme", "UMAP Component 1 (Semantic Similarity Axis)", "UMAP Component 2 (Theme Variation Axis)"]
    #assignments = viz_df[keep_base].copy()
    assignments = viz_df.copy()
# elif "clustering_results" in globals() and isinstance(clustering_results, pd.DataFrame) and len(clustering_results):
#     assignments = clustering_results.copy()
# elif Path("clustering_results.csv").exists():
#     assignments = pd.read_csv("clustering_results.csv")

# # --- Require clustering_results for the cluster map ---
# if "clustering_results" not in globals() or not isinstance(clustering_results, pd.DataFrame) or clustering_results.empty:
#     raise RuntimeError("clustering_results is missing/empty. Run the clustering cells before exporting.")

# assignments = clustering_results.copy()

if assignments.empty:
    raise RuntimeError(
        "No clustering dataframe found. Expected viz_df or clustering_results to exist "
        "before exporting artifacts."
    )

#  df
'''
restaurant name', 'UMAP Component 1 (Semantic Similarity Axis)',
       'UMAP Component 2 (Theme Variation Axis)', 'cluster', 'Primary Theme',
       'Number of Posts', 'Primary Theme %', 'All Themes'],
      dtype='str')
      '''

# Standardize columns
rename_map = {
    "input_restaurant_name": "name",
    "UMAP Component 1 (Semantic Similarity Axis)": "x",
    "UMAP Component 2 (Theme Variation Axis)": "y",
    'cluster': 'cluster_id',
    'theme': 'cluster_label',
    # Allow already-standard columns to pass through
}

assignments = assignments.rename(columns={k: v for k, v in rename_map.items() if k in assignments.columns})

# Keep ONLY the columns the dashboard needs (prevents duplicate-column explosions)
keep_base = ["name", "cluster_id", "cluster_label", "x", "y"]
missing = [c for c in keep_base if c not in assignments.columns]
if missing:
    raise RuntimeError(f"Missing required columns in viz_df/viz_df: {missing}")

# Ensure required columns exist
if "name" not in assignments.columns:
    # Try alternative source column
    if "restaurant name" in assignments.columns:
        assignments["name"] = assignments["restaurant name"]
    else:
        raise RuntimeError("Assignments missing restaurant name column (expected 'name' or 'restaurant name').")

if "cluster_id" not in assignments.columns:
    # If cluster id isn't present but label exists, create a dummy cluster id
    assignments["cluster_id"] = pd.factorize(assignments.get("cluster_label", assignments["name"]))[0]

if "cluster_label" not in assignments.columns:
    assignments["cluster_label"] = assignments["cluster_id"].apply(lambda v: f"Cluster {v}")

# Ensure x/y exist (cluster map requires them)
if "x" not in assignments.columns or "y" not in assignments.columns:
    # If x/y are missing, create a simple placeholder embedding to avoid breaking the dashboard
    # (Better than failing; but you should still export viz_df next run.)
    rng = np.random.default_rng(42)
    if "x" not in assignments.columns:
        assignments["x"] = rng.normal(loc=0.0, scale=1.0, size=len(assignments))
    if "y" not in assignments.columns:
        assignments["y"] = rng.normal(loc=0.0, scale=1.0, size=len(assignments))
    print("⚠️  x/y UMAP coordinates missing. Exported placeholder coordinates. "
          "To fix properly, ensure viz_df is created before this cell runs.")

# Attach restaurant_id if possible (needed for joining to other modules)
# Prefer clustering_results merge output if it exists; else try to merge with rest dataframe.
if "restaurant_id" not in assignments.columns:
    if "rest" in globals() and isinstance(rest, pd.DataFrame) and "name" in rest.columns and "restaurant_id" in rest.columns:
        tmp = rest[["restaurant_id", "name"]].drop_duplicates("name")
        assignments = assignments.merge(tmp, on="name", how="left")
    else:
        assignments["restaurant_id"] = np.nan

# Normalize types
# Check if cluster_id contains strings that aren't purely numeric
if assignments["cluster_id"].dtype == object:
    # Try to extract just the numbers (e.g., "Cluster 3" -> 3)
    extracted = assignments["cluster_id"].astype(str).str.extract(r'(\d+)')[0]
    
    # If extraction failed (e.g., they were labeled "A", "B"), use factorize
    if extracted.isnull().all():
        assignments["cluster_id"] = pd.factorize(assignments["cluster_id"])[0]
    else:
        assignments["cluster_id"] = extracted

# Now safely convert to int
assignments["cluster_id"] = pd.to_numeric(assignments["cluster_id"], errors="coerce").fillna(-1).astype(int)
assignments["restaurant_id"] = pd.to_numeric(assignments["restaurant_id"], errors="coerce")

# Keep only what the dashboard needs
assignments_out = assignments[[
    c for c in ["restaurant_id", "name", "cluster_id", "cluster_label", "x", "y"]
    if c in assignments.columns
]].copy()
print(f"Assignments has {assignments['cluster_id'].isnull().sum()} null cluster values")
assignments_out = assignments[[
    "restaurant_id","name","cluster_id","cluster_label","x","y"
]].copy()

# Add optional fields expected by some loaders/pages
assignments_out["cluster_confidence"] = np.nan
assignments_out["year_month"] = pd.NaT

print(f"Assignment out has {assignments_out['cluster_label'].isnull().sum()} null cluster values")
assignments_path = export_dir / "restaurant_cluster_assignments.parquet"
assignments_out.to_parquet(assignments_path, index=False)
print(f"Saved: {assignments_path} ({len(assignments_out):,} rows)")

# -----------------------------
# 2) cluster_keywords.parquet
# -----------------------------
# Build keywords per cluster from TF-IDF centroids (more meaningful than using theme_counts)
keywords_rows = []
if "kmeans" in globals() and "vectorizer" in globals():
    feature_names = vectorizer.get_feature_names_out()
    centroids = kmeans.cluster_centers_
    k = centroids.shape[0]
    for cid in range(k):
        top_idx = np.argsort(centroids[cid])[-50:][::-1]
        for rank, idx in enumerate(top_idx, start=1):
            keywords_rows.append({
                "cluster_id": int(cid),
                "keyword": str(feature_names[idx]),
                "weight": float(centroids[cid][idx]),
                "rank": int(rank),
            })
else:
    # Fallback: token frequency from raw text if kmeans/vectorizer not present
    tmp_reviews = reviews.copy() if "reviews" in globals() else pd.DataFrame()
    if not tmp_reviews.empty and "cluster" in tmp_reviews.columns and "text" in tmp_reviews.columns:
        tmp_reviews = tmp_reviews.rename(columns={"cluster": "cluster_id", "text": "raw_text"})
        for cid, cdf in tmp_reviews.groupby("cluster_id"):
            tokens = (
                cdf["raw_text"].fillna("").astype(str)
                .str.lower()
                .str.findall(r"[a-zA-Z]{3,}")
                .explode()
                .dropna()
            )
            top_terms = tokens.value_counts().head(50)
            for rank, (word, weight) in enumerate(top_terms.items(), start=1):
                keywords_rows.append({"cluster_id": int(cid), "keyword": word, "weight": float(weight), "rank": int(rank)})

keywords = pd.DataFrame(keywords_rows, columns=["cluster_id", "keyword", "weight", "rank"])
keywords_path = export_dir / "cluster_keywords.parquet"
keywords.to_parquet(keywords_path, index=False)
print(f"Saved: {keywords_path} ({len(keywords):,} rows)")

# -----------------------------
# 3) restaurant_text_corpus.parquet
# -----------------------------
# Keep per-restaurant text corpus + cluster assignment (useful for search/debug)
if "reviews" in globals() and isinstance(reviews, pd.DataFrame) and len(reviews):
    text_corpus = reviews.copy()
else:
    text_corpus = pd.DataFrame(columns=["restaurant name", "text", "cluster"])

text_corpus = text_corpus.rename(columns={
    "restaurant name": "name",
    "text": "raw_text",
    "text_clean": "clean_text",
    "cluster": "cluster_id",
})
if "text_id" not in text_corpus.columns:
    text_corpus["text_id"] = np.arange(1, len(text_corpus) + 1)

# Ensure required fields
if "clean_text" not in text_corpus.columns:
    text_corpus["clean_text"] = (
        text_corpus["raw_text"].fillna("").astype(str)
        .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    )
if "year_month" not in text_corpus.columns:
    text_corpus["year_month"] = pd.NaT

text_cols = [c for c in ["name", "text_id", "raw_text", "clean_text", "cluster_id", "year_month"] if c in text_corpus.columns]
text_corpus = text_corpus[text_cols].copy()

text_path = export_dir / "restaurant_text_corpus.parquet"
text_corpus.to_parquet(text_path, index=False)
print(f"Saved: {text_path} ({len(text_corpus):,} rows)")

# -----------------------------
# 4) cluster_strategy_outcomes.parquet
# -----------------------------
# Optional: only if marketing uplift file exists
marketing_path = project_root / "_3_marketing" / "activity_performance_with_roi.csv"
if marketing_path.exists() and assignments_out["restaurant_id"].notna().any():
    marketing = pd.read_csv(marketing_path)
    marketing["restaurant_id"] = pd.to_numeric(marketing.get("restaurant_id"), errors="coerce")

    assign_join = assignments_out[["restaurant_id", "name", "cluster_id", "cluster_label"]].dropna(subset=["restaurant_id"]).copy()
    assign_join["restaurant_id"] = assign_join["restaurant_id"].astype(int)

    strategy = marketing.merge(assign_join, on="restaurant_id", how="inner")

    # Basic computed fields used by downstream strategy logic
    strategy["roi"] = pd.to_numeric(strategy.get("roi"), errors="coerce")
    strategy["applied_date"] = pd.to_datetime(strategy.get("activity_start"), errors="coerce")
    strategy["sample_size"] = 1

    strategy_path = export_dir / "cluster_strategy_outcomes.parquet"
    strategy.to_parquet(strategy_path, index=False)
    print(f"Saved: {strategy_path} ({len(strategy):,} rows)")
else:
    strategy_path = export_dir / "cluster_strategy_outcomes.parquet"
    pd.DataFrame().to_parquet(strategy_path, index=False)
    print(f"Saved: {strategy_path} (0 rows; marketing file missing or no restaurant_id mapping)")


STEP 10: Export dashboard artifacts for Streamlit (UPDATED)
Assignments has 0 null cluster values
Assignment out has 0 null cluster values
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/restaurant_cluster_assignments.parquet (1,950 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/cluster_keywords.parquet (250 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/restaurant_text_corpus.parquet (1,950 rows)
Saved: /Users/jadentyh/Desktop/IS455/Project/OPE/frontend_dashboard/data/clustering/cluster_strategy_outcomes.parquet (1,124 rows)


In [26]:
import pandas as pd

assign = pd.read_parquet("../frontend_dashboard/data/clustering/restaurant_cluster_assignments.parquet")
uncl = assign[assign["cluster_id"] == -1].copy()

print("Unclustered rows:", len(uncl))
print(uncl[["restaurant_id", "name", "cluster_label"]].head(20))

Unclustered rows: 0
Empty DataFrame
Columns: [restaurant_id, name, cluster_label]
Index: []
